# CodonFM SAE — Codon Analysis

For each SAE feature, we compute codon-level properties of the positions where it activates. This reveals whether features track amino acid identity, synonymous codon preferences, codon usage bias, or positional patterns.

We compute three standard codon optimality metrics:
- **CAI (Codon Adaptation Index)** — how well codons match the usage pattern of highly expressed genes. High CAI = optimized for efficient translation.
- **tAI (tRNA Adaptation Index)** — how well codons match tRNA availability. High tAI = faster translation due to abundant cognate tRNAs.
- **RSCU (Relative Synonymous Codon Usage)** — how biased the synonymous codon choice is. RSCU=1 means no preference among synonyms; >1 means the common synonym is preferred.

## Setup

In [ ]:
SAE_CHECKPOINT = "../outputs/1b_layer16/checkpoints/checkpoint_final.pt"
MODEL_PATH = "../../../../../../../checkpoints/NV-CodonFM-Encodon-TE-Cdwt-1B-v1"
CSV_PATH = "../../../../../../../codonfm/data/codonfm_ref_only.csv"
LAYER = 16
CONTEXT_LENGTH = 2048
BATCH_SIZE = 8
NUM_SEQUENCES = 2000
DEVICE = "cuda"

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm


_REPO_ROOT = Path("..").resolve().parent.parent.parent.parent.parent
_CODONFM_TE_DIR = _REPO_ROOT / "recipes" / "codonfm_ptl_te"
sys.path.insert(0, str(_CODONFM_TE_DIR))
sys.path.insert(0, str(Path("..").resolve()))

from codonfm_sae.data import read_codon_csv
from sae.architectures import TopKSAE
from sae.utils import set_seed
from src.data.preprocess.codon_sequence import process_item
from src.inference.encodon import EncodonInference


set_seed(42)
device = DEVICE if torch.cuda.is_available() else "cpu"

## Load SAE, Model, and Data

In [ ]:
# Load SAE
ckpt = torch.load(SAE_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = ckpt["model_state_dict"]
if any(k.startswith("module.") for k in state_dict):
    state_dict = {k.removeprefix("module."): v for k, v in state_dict.items()}
input_dim = ckpt.get("input_dim") or state_dict["encoder.weight"].shape[1]
hidden_dim = ckpt.get("hidden_dim") or state_dict["encoder.weight"].shape[0]
model_config = ckpt.get("model_config", {})
sae = TopKSAE(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    top_k=model_config.get("top_k"),
    normalize_input=model_config.get("normalize_input", False),
)
sae.load_state_dict(state_dict)
sae = sae.eval().to(device)
print(f"SAE: {input_dim} \u2192 {hidden_dim:,} features")

# Load Encodon
inference = EncodonInference(model_path=MODEL_PATH, task_type="embedding_prediction", use_transformer_engine=True)
inference.configure_model()
inference.model.to(device).eval()
print(f"Encodon loaded ({len(inference.model.model.layers)} layers)")

# Load data
records = read_codon_csv(CSV_PATH, max_sequences=NUM_SEQUENCES, max_codons=CONTEXT_LENGTH - 2)
sequences = [r.sequence for r in records]
print(f"Loaded {len(sequences)} sequences")

## Codon Optimality Reference Tables

We use three reference tables to compute per-codon optimality weights:
- **Human codon usage frequencies** (Kazusa database) for CAI
- **Human tRNA gene copy numbers** (GtRNAdb, hg38) for tAI
- **RSCU** derived from the usage frequencies

For each metric, the weight of a codon is normalized relative to the best synonym for the same amino acid.

In [ ]:
HUMAN_CODON_USAGE = {
    "TTT": 17.6,
    "TTC": 20.3,
    "TTA": 7.7,
    "TTG": 12.9,
    "CTT": 13.2,
    "CTC": 19.6,
    "CTA": 7.2,
    "CTG": 39.6,
    "ATT": 16.0,
    "ATC": 20.8,
    "ATA": 7.5,
    "ATG": 22.0,
    "GTT": 11.0,
    "GTC": 14.5,
    "GTA": 7.1,
    "GTG": 28.1,
    "TCT": 15.2,
    "TCC": 17.7,
    "TCA": 12.2,
    "TCG": 4.4,
    "CCT": 17.5,
    "CCC": 19.8,
    "CCA": 16.9,
    "CCG": 6.9,
    "ACT": 13.1,
    "ACC": 18.9,
    "ACA": 15.1,
    "ACG": 6.1,
    "GCT": 18.4,
    "GCC": 27.7,
    "GCA": 15.8,
    "GCG": 7.4,
    "TAT": 12.2,
    "TAC": 15.3,
    "TAA": 1.0,
    "TAG": 0.8,
    "CAT": 10.9,
    "CAC": 15.1,
    "CAA": 12.3,
    "CAG": 34.2,
    "AAT": 17.0,
    "AAC": 19.1,
    "AAA": 24.4,
    "AAG": 31.9,
    "GAT": 21.8,
    "GAC": 25.1,
    "GAA": 29.0,
    "GAG": 39.6,
    "TGT": 10.6,
    "TGC": 12.6,
    "TGA": 1.6,
    "TGG": 13.2,
    "CGT": 4.5,
    "CGC": 10.4,
    "CGA": 6.2,
    "CGG": 11.4,
    "AGT": 12.1,
    "AGC": 19.5,
    "AGA": 12.2,
    "AGG": 12.0,
    "GGT": 10.8,
    "GGC": 22.2,
    "GGA": 16.5,
    "GGG": 16.5,
}

CODON_TO_AA = {
    "TTT": "F",
    "TTC": "F",
    "TTA": "L",
    "TTG": "L",
    "CTT": "L",
    "CTC": "L",
    "CTA": "L",
    "CTG": "L",
    "ATT": "I",
    "ATC": "I",
    "ATA": "I",
    "ATG": "M",
    "GTT": "V",
    "GTC": "V",
    "GTA": "V",
    "GTG": "V",
    "TCT": "S",
    "TCC": "S",
    "TCA": "S",
    "TCG": "S",
    "CCT": "P",
    "CCC": "P",
    "CCA": "P",
    "CCG": "P",
    "ACT": "T",
    "ACC": "T",
    "ACA": "T",
    "ACG": "T",
    "GCT": "A",
    "GCC": "A",
    "GCA": "A",
    "GCG": "A",
    "TAT": "Y",
    "TAC": "Y",
    "TAA": "*",
    "TAG": "*",
    "CAT": "H",
    "CAC": "H",
    "CAA": "Q",
    "CAG": "Q",
    "AAT": "N",
    "AAC": "N",
    "AAA": "K",
    "AAG": "K",
    "GAT": "D",
    "GAC": "D",
    "GAA": "E",
    "GAG": "E",
    "TGT": "C",
    "TGC": "C",
    "TGA": "*",
    "TGG": "W",
    "CGT": "R",
    "CGC": "R",
    "CGA": "R",
    "CGG": "R",
    "AGT": "S",
    "AGC": "S",
    "AGA": "R",
    "AGG": "R",
    "GGT": "G",
    "GGC": "G",
    "GGA": "G",
    "GGG": "G",
}

_HUMAN_TRNA_COPY_NUMBERS = {
    "TTT": 10,
    "TTC": 20,
    "TTA": 6,
    "TTG": 11,
    "CTT": 10,
    "CTC": 20,
    "CTA": 5,
    "CTG": 20,
    "ATT": 15,
    "ATC": 23,
    "ATA": 5,
    "ATG": 23,
    "GTT": 11,
    "GTC": 14,
    "GTA": 5,
    "GTG": 16,
    "TCT": 11,
    "TCC": 17,
    "TCA": 7,
    "TCG": 4,
    "CCT": 10,
    "CCC": 12,
    "CCA": 13,
    "CCG": 5,
    "ACT": 10,
    "ACC": 20,
    "ACA": 10,
    "ACG": 6,
    "GCT": 16,
    "GCC": 34,
    "GCA": 10,
    "GCG": 6,
    "TAT": 10,
    "TAC": 16,
    "TAA": 0,
    "TAG": 0,
    "CAT": 10,
    "CAC": 15,
    "CAA": 10,
    "CAG": 34,
    "AAT": 14,
    "AAC": 20,
    "AAA": 15,
    "AAG": 34,
    "GAT": 17,
    "GAC": 25,
    "GAA": 16,
    "GAG": 40,
    "TGT": 10,
    "TGC": 20,
    "TGA": 0,
    "TGG": 10,
    "CGT": 6,
    "CGC": 15,
    "CGA": 5,
    "CGG": 5,
    "AGT": 8,
    "AGC": 18,
    "AGA": 10,
    "AGG": 8,
    "GGT": 10,
    "GGC": 22,
    "GGA": 10,
    "GGG": 8,
}

from collections import defaultdict


aa_codons = defaultdict(list)
for codon, aa in CODON_TO_AA.items():
    if aa != "*":
        aa_codons[aa].append(codon)

CAI_WEIGHTS = {}
for aa, codons in aa_codons.items():
    freqs = [HUMAN_CODON_USAGE.get(c, 0.0) for c in codons]
    max_freq = max(freqs)
    for c, f in zip(codons, freqs):
        CAI_WEIGHTS[c] = f / max_freq if max_freq > 0 else 0.0

RSCU_VALUES = {}
for aa, codons in aa_codons.items():
    freqs = [HUMAN_CODON_USAGE.get(c, 0.0) for c in codons]
    total = sum(freqs)
    n_syn = len(codons)
    for c, f in zip(codons, freqs):
        RSCU_VALUES[c] = (f * n_syn / total) if total > 0 else 1.0

TAI_WEIGHTS = {}
for aa, codons in aa_codons.items():
    copies = [_HUMAN_TRNA_COPY_NUMBERS.get(c, 0) for c in codons]
    max_copy = max(copies)
    for c, cp in zip(codons, copies):
        TAI_WEIGHTS[c] = cp / max_copy if max_copy > 0 else 0.0

print(f"Built weights for {len(CAI_WEIGHTS)} codons")
print(
    f"Example \u2014 CTG(Leu): CAI={CAI_WEIGHTS['CTG']:.3f}, tAI={TAI_WEIGHTS['CTG']:.3f}, RSCU={RSCU_VALUES['CTG']:.3f}"
)
print(
    f"Example \u2014 CTA(Leu): CAI={CAI_WEIGHTS['CTA']:.3f}, tAI={TAI_WEIGHTS['CTA']:.3f}, RSCU={RSCU_VALUES['CTA']:.3f}"
)

## Streaming Codon Analysis

We stream sequences through the model and SAE, accumulating per-feature statistics:
- **Amino acid identity**: which amino acid each feature fires on most
- **Codon usage bias**: rare vs common codons
- **Wobble position**: GC vs AT at the 3rd (wobble) position
- **CpG sites**: whether the feature tracks CpG dinucleotides across codon boundaries
- **CAI/tAI/RSCU**: codon optimality metrics (geometric mean for CAI/tAI, arithmetic mean for RSCU)

In [ ]:
n_features = sae.hidden_dim
all_aas = sorted(set(CODON_TO_AA.values()))
aa_to_idx = {aa: i for i, aa in enumerate(all_aas)}
n_aa = len(all_aas)

# Accumulators
aa_counts = np.zeros((n_aa, n_features), dtype=np.int64)
rare_counts = np.zeros(n_features, dtype=np.int64)
common_counts = np.zeros(n_features, dtype=np.int64)
wobble_gc_counts = np.zeros(n_features, dtype=np.int64)
wobble_at_counts = np.zeros(n_features, dtype=np.int64)
cai_log_sum = np.zeros(n_features, dtype=np.float64)
tai_log_sum = np.zeros(n_features, dtype=np.float64)
rscu_sum = np.zeros(n_features, dtype=np.float64)
optimality_count = np.zeros(n_features, dtype=np.int64)

n_batches = (len(sequences) + BATCH_SIZE - 1) // BATCH_SIZE

with torch.no_grad():
    for batch_start in tqdm(range(0, len(sequences), BATCH_SIZE), total=n_batches, desc="Streaming"):
        batch_seqs = sequences[batch_start : batch_start + BATCH_SIZE]
        items = [process_item(s, context_length=CONTEXT_LENGTH, tokenizer=inference.tokenizer) for s in batch_seqs]
        batch = {
            "input_ids": torch.tensor(np.stack([it["input_ids"] for it in items])).to(device),
            "attention_mask": torch.tensor(np.stack([it["attention_mask"] for it in items])).to(device),
        }
        out = inference.model(batch, return_hidden_states=True)
        hidden = out.all_hidden_states[LAYER].float()
        attn = batch["attention_mask"]

        # Build mask excluding CLS/SEP
        keep = attn.clone()
        keep[:, 0] = 0
        lengths = attn.sum(dim=1)
        for b in range(keep.shape[0]):
            sep = int(lengths[b].item()) - 1
            if sep > 0:
                keep[b, sep] = 0

        for b in range(len(batch_seqs)):
            vl = int(keep[b].sum().item())
            if vl == 0:
                continue
            emb = hidden[b, :vl, :]
            _, codes = sae(emb)
            codes_cpu = codes.cpu().numpy()

            seq = batch_seqs[b]
            codons = [seq[j * 3 : (j + 1) * 3].upper() for j in range(vl)]
            active = codes_cpu > 0

            # Amino acid counts
            aa_idx = np.array([aa_to_idx.get(CODON_TO_AA.get(c, "?"), 0) for c in codons])
            for a in range(n_aa):
                mask = aa_idx == a
                if mask.any():
                    aa_counts[a] += active[mask].sum(axis=0)

            # Usage bias
            is_rare = np.array([HUMAN_CODON_USAGE.get(c, 10.0) < 10.0 for c in codons])
            rare_counts += active[is_rare].sum(axis=0) if is_rare.any() else 0
            common_counts += active[~is_rare].sum(axis=0) if (~is_rare).any() else 0

            # Wobble
            is_gc = np.array([c[2] in ("G", "C") if len(c) == 3 else False for c in codons])
            wobble_gc_counts += active[is_gc].sum(axis=0) if is_gc.any() else 0
            wobble_at_counts += active[~is_gc].sum(axis=0) if (~is_gc).any() else 0

            # CAI/tAI/RSCU
            non_stop = np.array([CODON_TO_AA.get(c, "*") != "*" for c in codons])
            cai_w = np.array([CAI_WEIGHTS.get(c, 0.0) for c in codons])
            tai_w = np.array([TAI_WEIGHTS.get(c, 0.0) for c in codons])
            rscu_v = np.array([RSCU_VALUES.get(c, 1.0) for c in codons])

            valid_cai = non_stop & (cai_w > 0)
            valid_tai = non_stop & (tai_w > 0)

            if valid_cai.any():
                log_cai = np.log(cai_w[valid_cai])
                cai_log_sum += (active[valid_cai] * log_cai[:, None]).sum(axis=0)
            if valid_tai.any():
                log_tai = np.log(tai_w[valid_tai])
                tai_log_sum += (active[valid_tai] * log_tai[:, None]).sum(axis=0)
            if non_stop.any():
                rscu_sum += (active[non_stop] * rscu_v[non_stop, None]).sum(axis=0)
                optimality_count += active[non_stop].sum(axis=0)

        del out, hidden, batch
        torch.cuda.empty_cache()

print("Streaming complete.")

## Per-Feature Optimality Metrics

Now we compute the final CAI, tAI, and RSCU for each feature. CAI and tAI use the geometric mean (exp of mean log-weights), while RSCU uses the arithmetic mean.

In [ ]:
alive_mask = optimality_count > 10
n_alive_opt = alive_mask.sum()

feature_cai = np.full(n_features, np.nan)
feature_tai = np.full(n_features, np.nan)
feature_rscu = np.full(n_features, np.nan)

feature_cai[alive_mask] = np.exp(cai_log_sum[alive_mask] / optimality_count[alive_mask])
feature_tai[alive_mask] = np.exp(tai_log_sum[alive_mask] / optimality_count[alive_mask])
feature_rscu[alive_mask] = rscu_sum[alive_mask] / optimality_count[alive_mask]

print(f"Features with optimality metrics: {n_alive_opt:,}")
print(
    f"\nCAI:  mean={np.nanmean(feature_cai):.4f}, std={np.nanstd(feature_cai):.4f}, range=[{np.nanmin(feature_cai):.4f}, {np.nanmax(feature_cai):.4f}]"
)
print(
    f"tAI:  mean={np.nanmean(feature_tai):.4f}, std={np.nanstd(feature_tai):.4f}, range=[{np.nanmin(feature_tai):.4f}, {np.nanmax(feature_tai):.4f}]"
)
print(
    f"RSCU: mean={np.nanmean(feature_rscu):.4f}, std={np.nanstd(feature_rscu):.4f}, range=[{np.nanmin(feature_rscu):.4f}, {np.nanmax(feature_rscu):.4f}]"
)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, vals, name, color in [
    (axes[0], feature_cai, "CAI", "#76b900"),
    (axes[1], feature_tai, "tAI", "#0074DF"),
    (axes[2], feature_rscu, "RSCU", "#9525C6"),
]:
    valid = vals[~np.isnan(vals)]
    ax.hist(valid, bins=50, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(np.median(valid), color="red", linestyle="--", label=f"Median: {np.median(valid):.3f}")
    ax.set_xlabel(name)
    ax.set_ylabel("Number of features")
    ax.set_title(f"{name} Distribution")
    ax.legend()

plt.tight_layout()
plt.show()

## CAI vs tAI Correlation

CAI and tAI measure related but distinct properties. CAI reflects codon usage in highly expressed genes; tAI reflects tRNA availability. They're correlated (both favor 'optimal' codons) but can diverge \u2014 some codons have high usage but low tRNA counts.

In [ ]:
valid = alive_mask
fig, ax = plt.subplots(figsize=(6, 6))
sc = ax.scatter(feature_cai[valid], feature_tai[valid], c=feature_rscu[valid], s=3, alpha=0.3, cmap="viridis")
ax.set_xlabel("CAI (Codon Adaptation Index)")
ax.set_ylabel("tAI (tRNA Adaptation Index)")
ax.set_title("CAI vs tAI per Feature (colored by RSCU)")
plt.colorbar(sc, label="RSCU")
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
plt.tight_layout()
plt.show()

## Extreme Features

Features with unusually high or low optimality scores are the most interesting:
- **High CAI features** likely fire on codons in highly expressed, translationally optimized genes
- **Low CAI features** may track rare codons used in tissue-specific or lowly expressed genes
- **High RSCU features** strongly prefer the dominant synonymous codon
- **Low RSCU features** prefer the rare synonym \u2014 potentially detecting regulatory signals encoded in codon choice

In [ ]:
for metric, vals, name in [("CAI", feature_cai, "CAI"), ("tAI", feature_tai, "tAI"), ("RSCU", feature_rscu, "RSCU")]:
    valid_idx = np.where(~np.isnan(vals))[0]
    sorted_idx = valid_idx[np.argsort(vals[valid_idx])]

    print(f"\n{'=' * 50}")
    print(f"Top 5 highest {name}:")
    for i in sorted_idx[-5:][::-1]:
        # Find dominant amino acid
        total = aa_counts[:, i].sum()
        if total > 0:
            best_aa = all_aas[aa_counts[:, i].argmax()]
            aa_frac = aa_counts[:, i].max() / total
            aa_str = f"AA={best_aa} ({aa_frac:.0%})"
        else:
            aa_str = "no data"
        print(f"  Feature {i:>5d}: {name}={vals[i]:.4f}  {aa_str}")

    print(f"\nTop 5 lowest {name}:")
    for i in sorted_idx[:5]:
        total = aa_counts[:, i].sum()
        if total > 0:
            best_aa = all_aas[aa_counts[:, i].argmax()]
            aa_frac = aa_counts[:, i].max() / total
            aa_str = f"AA={best_aa} ({aa_frac:.0%})"
        else:
            aa_str = "no data"
        print(f"  Feature {i:>5d}: {name}={vals[i]:.4f}  {aa_str}")

## Save Results

Save the per-feature optimality metrics for use in the dashboard (notebook 04) or further analysis.

In [ ]:
output_dir = Path("../outputs/1b_layer16/analysis")
output_dir.mkdir(parents=True, exist_ok=True)

results = {}
for f in range(n_features):
    if alive_mask[f]:
        results[f] = {
            "cai": round(float(feature_cai[f]), 4),
            "tai": round(float(feature_tai[f]), 4),
            "rscu": round(float(feature_rscu[f]), 4),
        }

with open(output_dir / "codon_optimality.json", "w") as fp:
    json.dump(results, fp, indent=2)

print(f"Saved optimality metrics for {len(results)} features to {output_dir / 'codon_optimality.json'}")